In [5]:
import pandas as pd

df = pd.DataFrame({
    "y_true": [0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1],
    "y_pred": [0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1]
})

df.to_csv("evaluation_demo.csv", index=False)
print("CSV 파일 생성 완료!")

CSV 파일 생성 완료!


In [6]:
from google.colab import files
files.download("evaluation_demo.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
from infer_ci import MetricEvaluator

st.set_page_config(page_title="Evaluation Demo with Confidence Intervals", layout="centered")

st.title("📊 Model Evaluation Demo")
st.markdown("Upload a CSV file with **y_true** and **y_pred** columns to evaluate performance and view interactive bootstrap distributions.")

# 1. 파일 업로드 기능
uploaded_file = st.file_uploader("Upload your CSV file", type=["csv"])

if uploaded_file is not None:
    df = pd.read_csv(uploaded_file)
    st.subheader("Data Preview")
    st.dataframe(df.head())

    if "y_true" in df.columns and "y_pred" in df.columns:
        y_true = df["y_true"].values
        y_pred = df["y_pred"].values

        # 메트릭 선택
        metric = st.selectbox("Select Metric", ["accuracy", "precision", "recall", "f1"])

        evaluator = MetricEvaluator()

        with st.spinner("Computing metrics and bootstrap distribution..."):
            # 평가 수행
            value, ci = evaluator.evaluate(
                y_true=y_true,
                y_pred=y_pred,
                task='classification',
                metric=metric,
                method='bootstrap_bca',
                n_resamples=500
            )

            # 결과 표시
            st.markdown("---")
            st.subheader("Results")
            col1, col2 = st.columns(2)
            with col1:
                st.metric(label=f"Metric ({metric.capitalize()})", value=f"{value:.4f}")
            with col2:
                st.metric(label="95% Confidence Interval", value=f"[{ci[0]:.4f}, {ci[1]:.4f}]")

            # 부트스트랩 분포 시뮬레이션 데이터 생성 (시각화용)
            np.random.seed(42)
            boot_scores = []
            n = len(y_true)
            for _ in range(300):
                indices = np.random.choice(n, size=n, replace=True)
                yt_b = y_true[indices]
                yp_b = y_pred[indices]
                try:
                    val, _ = evaluator.evaluate(y_true=yt_b, y_pred=yp_b, task='classification', metric=metric, method='wilson' if metric=='accuracy' else 'bootstrap_bca')
                    boot_scores.append(val)
                except:
                    continue

            # 2. 결과 시각화 개선: Plotly를 이용한 인터랙티브 히스토그램 (툴팁 지원)
            st.markdown("### Bootstrap Distribution")
            st.markdown("💡 그래프의 막대에 마우스를 올리면 정확한 구간(Bin)과 빈도 수(Count)를 확인할 수 있습니다.")

            fig = px.histogram(
                x=boot_scores,
                nbins=12,
                labels={'x': 'Metric Score', 'y': 'Frequency'},
                color_discrete_sequence=['#f6ad55']
            )

            # 신뢰 구간 경계선 표시
            fig.add_vline(x=ci[0], line_dash="dash", line_color="red", annotation_text="Lower CI")
            fig.add_vline(x=ci[1], line_dash="dash", line_color="red", annotation_text="Upper CI")

            fig.update_layout(
                margin=dict(l=20, r=20, t=30, b=20),
                paper_bgcolor='rgba(0,0,0,0)',
                plot_bgcolor='rgba(0,0,0,0)'
            )

            st.plotly_chart(fig, use_container_width=True)

    else:
        st.error("⚠️ CSV 파일에 'y_true'와 'y_pred' 컬럼이 반드시 포함되어 있어야 합니다.")
else:
    st.info("ℹ️ 시작하려면 좌측 또는 상단의 버튼을 눌러 CSV 파일을 업로드하세요.")

2026-10-27 19:44:41.092 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-27 19:44:41.093 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-27 19:44:41.094 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-27 19:44:41.095 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-27 19:44:41.096 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-27 19:44:41.096 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-27 19:44:41.098 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-27 19:44:41.099 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [4]:
Python
!pip install streamlit infer-ci plotly

In [8]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
from infer_ci import MetricEvaluator

st.set_page_config(page_title="Evaluation Demo with Confidence Intervals", layout="centered")

st.title("📊 Model Evaluation Demo")
st.markdown("Upload a CSV file with **y_true** and **y_pred** columns to evaluate performance and view interactive bootstrap distributions.")

uploaded_file = st.file_uploader("Upload your CSV file", type=["csv"])

if uploaded_file is not None:
    df = pd.read_csv(uploaded_file)
    st.subheader("Data Preview")
    st.dataframe(df.head())

    if "y_true" in df.columns and "y_pred" in df.columns:
        y_true = df["y_true"].values
        y_pred = df["y_pred"].values

        metric = st.selectbox("Select Metric", ["accuracy", "precision", "recall", "f1"])
        evaluator = MetricEvaluator()

        with st.spinner("Computing metrics and bootstrap distribution..."):
            value, ci = evaluator.evaluate(
                y_true=y_true,
                y_pred=y_pred,
                task='classification',
                metric=metric,
                method='bootstrap_bca',
                n_resamples=500
            )

            st.markdown("---")
            st.subheader("Results")
            col1, col2 = st.columns(2)
            with col1:
                st.metric(label=f"Metric ({metric.capitalize()})", value=f"{value:.4f}")
            with col2:
                st.metric(label="95% Confidence Interval", value=f"[{ci[0]:.4f}, {ci[1]:.4f}]")

            np.random.seed(42)
            boot_scores = []
            n = len(y_true)
            for _ in range(300):
                indices = np.random.choice(n, size=n, replace=True)
                yt_b = y_true[indices]
                yp_b = y_pred[indices]
                try:
                    val, _ = evaluator.evaluate(y_true=yt_b, y_pred=yp_b, task='classification', metric=metric, method='wilson' if metric=='accuracy' else 'bootstrap_bca')
                    boot_scores.append(val)
                except:
                    continue

            st.markdown("### Bootstrap Distribution")
            st.markdown("💡 그래프의 막대에 마우스를 올리면 정확한 구간(Bin)과 빈도 수(Count)를 확인할 수 있습니다.")

            fig = px.histogram(
                x=boot_scores,
                nbins=12,
                labels={'x': 'Metric Score', 'y': 'Frequency'},
                color_discrete_sequence=['#f6ad55']
            )
            fig.add_vline(x=ci[0], line_dash="dash", line_color="red", annotation_text="Lower CI")
            fig.add_vline(x=ci[1], line_dash="dash", line_color="red", annotation_text="Upper CI")
            fig.update_layout(
                margin=dict(l=20, r=20, t=30, b=20),
                paper_bgcolor='rgba(0,0,0,0)',
                plot_bgcolor='rgba(0,0,0,0)'
            )
            st.plotly_chart(fig, use_container_width=True)
    else:
        st.error("⚠️ CSV 파일에 'y_true'와 'y_pred' 컬럼이 반드시 포함되어 있어야 합니다.")
else:
    st.info("ℹ️ 시작하려면 좌측 또는 상단의 버튼을 눌러 CSV 파일을 업로드하세요.")

Writing app.py


In [11]:
!streamlit run app.py & npx -y localtunnel --port 8501

⠙⠹

⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸2026-10-27 19:49:55.916 Uvicorn server started on :::8501
⠼⠴⠦
  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://136.67.251.38:8501

⠧your url is: https://cuddly-days-beam.loca.lt
Bootstrap CI: 100% 516/516 [00:00<00:00, 24316.72it/s]
2026-10-27 19:52:33.518 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
Bootstrap CI: 100% 516/516 [00:00<00:00, 33823.45it/s]
2026-10-27 19:53:38.177 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
  Stopping...
^C
